# Volume 7: Harder problems — Hilbert spaces at work (Python companion)

Part A checks the numbers of the chapter, problem by problem. Part B is a laboratory: three experiments that push the problems further, with exercises first and
solutions at the end. We use `numpy` for linear algebra, `mpmath` for $\zeta$ and high precision, and `sympy` for primes. The whole notebook runs in a few minutes.

*Before running a cell, predict its output.*

In [1]:
import math, random, cmath
from math import log, sqrt, pi, exp, gcd, cos, sin
import numpy as np
import mpmath as mp
from sympy import primerange, divisor_sigma, mobius, totient
random.seed(1); rng = np.random.default_rng(1)
mp.mp.dps = 20
e = lambda x: np.exp(2j * np.pi * x)

## Part A: checks

**Problem 2.1 (Selberg).** Random points $\alpha_r$ with minimal distance $\delta$, random $a_n$: compare $\sum_r|S(\alpha_r)|^2$ with $\bigl(N+\frac{1+\log R}\delta\bigr)\sum|a_n|^2$, and with the
sharp large sieve constant $N-1+\frac1\delta$.

In [2]:
N, R = 60, 40
for trial in range(3):
    al = (np.arange(R) + 0.5 * rng.random(R)) / R; d = np.diff(np.r_[al, al[0] + 1]); delta = d.min()
    a = rng.normal(size=N) + 1j * rng.normal(size=N)
    n = np.arange(1, N + 1)
    S = np.array([np.sum(a * e(n * x)) for x in al])
    lhs = np.sum(abs(S) ** 2) / np.sum(abs(a) ** 2)
    print("delta=%.4f  ratio=%.2f  Selberg bound=%.2f  sharp=%.2f" % (delta, lhs, N + (1 + log(R)) / delta, N - 1 + 1 / delta))

delta=0.0140  ratio=39.88  Selberg bound=394.24  sharp=130.28
delta=0.0139  ratio=39.01  Selberg bound=397.38  sharp=130.95
delta=0.0152  ratio=36.60  Selberg bound=367.83  sharp=124.65


**Problems 2.2 and 2.3 (Welch, Gerzon).** The six diagonals of the icosahedron: all $|\cos|$ equal $\frac1{\sqrt5}$, the Welch bound $\frac{R-n}{n(R-1)}$ for $R=6$, $n=3$ is $\frac15$, and
$6=\frac{3\cdot4}2$. The Gram matrix of the $P_i=u_iu_i^{\mathsf T}$ is invertible.

In [3]:
ph = (1 + sqrt(5)) / 2
D = np.array([(0, 1, ph), (0, 1, -ph), (1, ph, 0), (1, -ph, 0), (ph, 0, 1), (-ph, 0, 1)]); D /= np.linalg.norm(D, axis=1)[:, None]
G = D @ D.T
print("off-diagonal |cos|:", sorted(set(np.round(abs(G[~np.eye(6, dtype=bool)]), 12))), " 1/sqrt5 =", 1 / sqrt(5))
print("Welch bound eps^2 >=", (6 - 3) / (3 * 5))
Pm = [np.outer(u, u) for u in D]
GP = np.array([[np.trace(A @ B) for B in Pm] for A in Pm])
print("eigenvalues of the Gram matrix of the P_i:", np.round(np.linalg.eigvalsh(GP), 6))

off-diagonal |cos|: [np.float64(0.4472135955)]  1/sqrt5 = 0.4472135954999579
Welch bound eps^2 >= 0.2
eigenvalues of the Gram matrix of the P_i: [0.8 0.8 0.8 0.8 0.8 2. ]


**Problem 2.4(c) (all characters).** For $q=101$ and random $a_n$, $n\le N=20$: $\sum_\chi|\sum a_n\chi(n)|^2=(q-1)A$. We build the characters from a primitive root.

In [4]:
q, N = 101, 20
g = next(g for g in range(2, q) if all(pow(g, (q - 1) // p, q) != 1 for p in (2, 5)))
ind = {pow(g, k, q): k for k in range(q - 1)}
a = rng.normal(size=N) + 1j * rng.normal(size=N)
tot = 0
for j in range(q - 1):
    chi = np.array([np.exp(2j * pi * j * ind[n] / (q - 1)) for n in range(1, N + 1)])
    tot += abs(np.sum(a * chi)) ** 2
print(tot, (q - 1) * np.sum(abs(a) ** 2))

4193.120932072294 4193.1209320722755


**Problem 2.5 (Ingham).** The transform $K(\lambda)=\frac{4\pi T\cos\lambda T}{\pi^2-4\lambda^2T^2}$ against numerical integration; then the inequality (b) for random frequencies with gap $\gamma=1$ and
$T=4$.

In [5]:
T = 2.0
for lam in (0.0, 0.3, 1.7, 5.0):
    num = mp.quad(lambda t: mp.cos(mp.pi * t / (2 * T)) * mp.cos(lam * t), [-T, T])
    print(lam, float(num), 4 * pi * T * cos(lam * T) / (pi ** 2 - 4 * lam ** 2 * T ** 2))
T, gam = 4.0, 1.0
lam = np.cumsum(gam + rng.random(15)); a = rng.normal(size=15) + 1j * rng.normal(size=15)
t = np.linspace(-T, T, 200001); f = (a[None, :] * np.exp(1j * np.outer(t, lam))).sum(1)
I = np.trapezoid(abs(f) ** 2, t) / np.sum(abs(a) ** 2)
print("ratio", I, " lower bound", 4 * T / pi * (1 - pi ** 2 / (gam ** 2 * T ** 2)), " upper bound", 8 * sqrt(2) * T / pi * (1 + pi ** 2 / (4 * gam ** 2 * T ** 2)))

0.0 2.5464790894703255 2.5464790894703255
0.3 2.460725966415395 2.4607259664153953
1.7 0.6680787600567089 0.668078760056709
5.0 0.05405415176710313 0.05405415176710313


ratio 7.698736256923272  lower bound 1.9513655253508577  upper bound 16.626502527592883


**Problem 2.6 (mean values).** (a) $\hat k$ is a triangle; (b) $\min w$ on $[0,T]$ against $1-\frac2{\pi^2\Delta U}$ for $\Delta=1$, $T=6$, $U=2$ (Figure 2.2); (c) the bound
$(T+5\sqrt{TN})\sum|a_n|^2$ for $N=40$, $T=16N$, with the worst vector, the top eigenvector of the matrix $\bigl(\int_0^T(m/n)^{it}\dd t\bigr)$.

In [6]:
k = lambda t: 1.0 if t == 0 else (math.sin(pi * t) / (pi * t)) ** 2
for xi in (0, 0.25, 0.5, 0.9, 1.3):
    v = mp.quad(lambda t: (mp.sinc(mp.pi * t)) ** 2 * mp.cos(2 * mp.pi * xi * t), mp.linspace(0, 400, 401)) * 2
    print("khat(%.2f) = %.5f  triangle %.5f" % (xi, v, max(0, 1 - xi)))
U, T = 2.0, 6.0
w = lambda t: float(mp.quad(lambda s: mp.sinc(mp.pi * (t - s)) ** 2, [-U, t - 1, t, t + 1, T + U]))
print("min w on [0,T] ~", min(w(t) for t in np.linspace(0, T, 61)), " bound", 1 - 2 / (pi ** 2 * U))
N = 40; T = 16 * N
L = np.log(np.arange(1, N + 1))
dl = L[:, None] - L[None, :]
with np.errstate(invalid="ignore", divide="ignore"):
    M = np.where(dl == 0, T, (np.exp(1j * dl * T) - 1) / (1j * dl))
lmax = np.linalg.eigvalsh((M + M.conj().T) / 2).max()
print("largest eigenvalue", lmax, " T =", T, " bound T+5sqrt(TN) =", T + 5 * sqrt(T * N))

khat(0.00) = 0.99975  triangle 1.00000


khat(0.25) = 0.75000  triangle 0.75000


khat(0.50) = 0.50000  triangle 0.50000


khat(0.90) = 0.10000  triangle 0.10000


khat(1.30) = 0.00000  triangle 0.00000


min w on [0,T] ~ 0.968642085061876  bound 0.8986788163576622
largest eigenvalue 768.2701198506556  T = 640  bound T+5sqrt(TN) = 1440.0


**Problem 2.7 (Christoffel).** $K_n(1,1)=\frac{(n+1)^2}2$, the extremal polynomial for $n=2$, $K_2(0,0)=\frac98$, and the interior asymptotics $K_n(x,x)\sim\frac n{\pi\sqrt{1-x^2}}$ quoted in the solution.

In [7]:
from numpy.polynomial import legendre as Lg
def K(n, x, y):
    return sum((k + 0.5) * Lg.legval(x, [0] * k + [1]) * Lg.legval(y, [0] * k + [1]) for k in range(n + 1))
print([K(n, 1, 1) for n in range(5)], [(n + 1) ** 2 / 2 for n in range(5)])
p = np.polynomial.Polynomial([-3 / 4, 6 / 4, 15 / 4]); q2 = (p * p).integ()
print("p(1)^2/int p^2 =", p(1) ** 2 / (q2(1) - q2(-1)), " K_2(0,0) =", K(2, 0, 0), 9 / 8)
for n in (200, 800):
    print(n, [round(K(n, x, x) / (n / (pi * sqrt(1 - x * x))), 4) for x in (0, 0.5, 0.9)])

[np.float64(0.5), np.float64(2.0), np.float64(4.5), np.float64(8.0), np.float64(12.500000000000004)] [0.5, 2.0, 4.5, 8.0, 12.5]
p(1)^2/int p^2 = 4.5  K_2(0,0) = 1.125 1.125
200 [np.float64(1.0075), np.float64(1.0021), np.float64(1.0015)]


800 [np.float64(1.0019), np.float64(1.0005), np.float64(0.9998)]


**Problems 2.8 and 2.9 (Bergman, Fock).** For $f(z)=\frac1{1-z/2}$, $\int_{\mathbb D}|f|^2=\pi\sum\frac{4^{-n}}{n+1}=4\pi\log\frac43$. And $\|e^{az^2}\|^2_{\mathcal F}=(1-4|a|^2)^{-1/2}$ for $a=0.3$, by
quadrature in polar coordinates.

In [8]:
I = mp.quad(lambda r, th: r * abs(1 / (1 - r * mp.exp(1j * th) / 2)) ** 2, [0, 1], [0, 2 * mp.pi])
print(I, 4 * mp.pi * mp.log(mp.mpf(4) / 3))
a = 0.3
I = mp.quad(lambda r, th: r * abs(mp.exp(a * (r * mp.exp(1j * th)) ** 2)) ** 2 * mp.exp(-r * r), [0, 3, 6, 12], [0, mp.pi / 2, mp.pi, 3 * mp.pi / 2, 2 * mp.pi]) / mp.pi
print(I, (1 - 4 * a * a) ** -0.5)

3.6151195415360063983 3.6151195415360063983


1.249999999999999974 1.25


**Problem 2.10 (Sobolev).** The reproducing property for $u(x)=x^2+\sin3x$ on $[0,L]$, $L=1$, and the constants $\coth1$ and $\frac12\coth\frac12$.

In [9]:
L_ = mp.mpf(1)
Ky = lambda y: (lambda x: mp.cosh(min(x, y)) * mp.cosh(L_ - max(x, y)) / mp.sinh(L_))
dKy = lambda y: (lambda x: (mp.sinh(x) * mp.cosh(L_ - y) if x < y else -mp.cosh(y) * mp.sinh(L_ - x)) / mp.sinh(L_))
u = lambda x: x ** 2 + mp.sin(3 * x); du = lambda x: 2 * x + 3 * mp.cos(3 * x)
for y in (mp.mpf(0), mp.mpf("0.3"), mp.mpf(1)):
    ip = mp.quad(lambda x: u(x) * Ky(y)(x) + du(x) * dKy(y)(x), [0, y, L_] if 0 < y < L_ else [0, L_])
    print(float(ip), float(u(y)))
print("coth 1 =", mp.coth(1), "  coth(1/2)/2 =", mp.coth(0.5) / 2, "  K_y(y) max over y:", max(float(mp.cosh(y) * mp.cosh(1 - y) / mp.sinh(1)) for y in np.linspace(0, 1, 101)))

2.0068173449603102e-25 0.0
0.8733269096274834 0.8733269096274834
1.1411200080598671 1.1411200080598671
coth 1 = 1.3130352854993313036   coth(1/2)/2 = 1.0819767068693264244   K_y(y) max over y: 1.3130352854993312


**Problem 2.11(c) (a norm in $\mathcal H$).** With a divisor sieve: $\sum_{n\le M}c_n^2$ against $\frac{\zeta(2a)\zeta(2b)\zeta(a+b)^2}{\zeta(2a+2b)}$ for $a=1.2$, $b=1.5$, and the identity
$\sum_kh_k^2=\frac{1+uv}{(1-u^2)(1-v^2)(1-uv)}$.

In [10]:
a_, b_, M = 1.2, 1.5, 10 ** 6
c = np.zeros(M + 1)
for d in range(1, M + 1):
    c[d::d] += d ** -a_ * (np.arange(1, M // d + 1)) ** -b_
print(np.sum(c[1:] ** 2), mp.zeta(2 * a_) * mp.zeta(2 * b_) * mp.zeta(a_ + b_) ** 2 / mp.zeta(2 * a_ + 2 * b_))
u, v = 0.37, 0.21
print(sum(((u ** (k + 1) - v ** (k + 1)) / (u - v)) ** 2 for k in range(200)), (1 + u * v) / ((1 - u * u) * (1 - v * v) * (1 - u * v)))

2.628604321469873 2.6286043940394856496
1.4162896975764603 1.4162896975764607


**Problem 2.12 (Franel).** The integral for $(m,n)=(4,6)$; then $S(N)-3N$ against $\log^2N$; and Euler's $\sum\frac{H_k}{k^2}=2\zeta(3)$.

In [11]:
B = lambda x: x - mp.floor(x) - 0.5
m, n = 4, 6
I = mp.quad(lambda x: B(m * x) * B(n * x), [j / 12 for j in range(13)])
print(I, gcd(m, n) ** 2 / (12 * m * n))
for N in (100, 1000, 3000):
    S = sum(gcd(m, n) ** 2 / (m * n) for m in range(1, N + 1) for n in range(1, N + 1))
    print(N, S - 3 * N, log(N) ** 2)
K = 10 ** 6; kk = np.arange(1, K + 1, dtype=float); H = np.cumsum(1 / kk)
print(np.sum(H / kk ** 2) + (log(K) + 1 + float(mp.euler)) / K, 2 * mp.zeta(3))      # tail ~ int (log x + gamma)/x^2

0.013888888888888918122 0.013888888888888888
100 -14.575000713540987 21.207592441913597
1000 -27.523681021122684 47.71708299430558


3000 -33.41539302458659 64.10192162832172
2.4041138063261354 2.4041138063191885708


**Problem 2.13 (Ramanujan).** The expansion at $n=12$ truncated at $q\le2000$, and the mean of $(\sigma(n)/n)^2$ for $N=2\cdot10^6$ against $\frac52\zeta(3)$.

In [12]:
def cq(q, n):
    return sum(int(mobius(q // d)) * d for d in range(1, q + 1) if q % d == 0 and n % d == 0)
n = 12
print(float(divisor_sigma(n)) / n, float(mp.zeta(2)) * sum(cq(q, n) / q ** 2 for q in range(1, 2001)))
N = 2 * 10 ** 6
sig = np.zeros(N + 1)
for d in range(1, N + 1):
    sig[d::d] += d
x = sig[1:] / np.arange(1, N + 1)
print(np.mean(x ** 2), 2.5 * mp.zeta(3))

2.3333333333333335 2.333335849037341


3.005104427534362 3.0051422578989857135


**Problem 2.14 (Plancherel on the critical line).** $\frac1{2\pi}\int\frac{|\zeta(1/2+it)|^2}{1/4+t^2}\dd t$, integrating up to $|t|=300$ and estimating the tail with the mean value
$|\zeta(\frac12+it)|^2\approx\log\frac t{2\pi}+2\gamma$; against $\log2\pi-\gamma$.

In [13]:
mp.mp.dps = 15
Tm = 300
I = 2 * mp.quad(lambda t: abs(mp.zeta(0.5 + 1j * t)) ** 2 / (0.25 + t * t), mp.linspace(0, Tm, 301))
tail = 2 * (mp.log(Tm / (2 * mp.pi)) + 1 + 2 * mp.euler) / Tm
print("integral up to 300:", I / (2 * mp.pi), " with tail:", (I + tail) / (2 * mp.pi), " log 2pi - gamma =", mp.log(2 * mp.pi) - mp.euler)

integral up to 300: 1.25427325172926  with tail: 1.26066102740219  log 2pi - gamma = 1.26066140150781


**Problem 2.15 (Nyman).** On $]0,1[$, $\rho_{1/k}(x)=\frac1k\lfloor\frac1x\rfloor-\lfloor\frac1{kx}\rfloor$ is a step function (check), and its Mellin transform at $s=0.7$ equals $-\frac{\zeta(s)}s(k^{-s}-\frac1k)$.

In [14]:
k, s = 3, mp.mpf("0.7")
x = mp.mpf("0.0173")
print(mp.frac(1 / (k * x)) - mp.frac(1 / x) / k, mp.floor(1 / x) / k - mp.floor(1 / (k * x)))
# in u = 1/x the integral is sum_n f_k(n) * int_n^{n+1} u^{-s-1} du with f_k(n) = (n mod k)/k
M = 10 ** 6
nn = np.arange(1, M + 1, dtype=float)
val = np.sum(((np.arange(1, M + 1) % k) / k) * (nn ** -0.7 - (nn + 1) ** -0.7) / 0.7) + ((k - 1) / (2 * k)) * M ** -0.7 / 0.7
print(val, -mp.zeta(s) / s * (mp.mpf(k) ** -s - mp.mpf(1) / k))

-3.5527136788005e-15 0.0
0.5165013143354577 0.516501314321437


**Problem 2.16 (Bohr).** $\zeta(2)=\frac{\pi^2}6$ and $\frac{\zeta(4)}{\zeta(2)}=\frac{\pi^2}{15}$; scanning $t\le2\cdot10^5$ for small $|\zeta(2+it)|$ (where $2^{-it}$, $3^{-it}$, $5^{-it}$, $7^{-it}$ are all near $-1$) gets
close to the infimum.

In [15]:
print(mp.zeta(2), mp.pi ** 2 / 6, mp.zeta(4) / mp.zeta(2), mp.pi ** 2 / 15)
t = np.arange(0, 2e5, 0.01)
score = sum(np.cos(t * log(p)) for p in (2, 3, 5, 7))      # -4 when all four phases are -1
cand = t[np.argsort(score)[:30]]
best = min((abs(mp.zeta(2 + 1j * float(x))), float(x)) for x in cand)
print("smallest |zeta(2+it)| found:", best)

1.64493406684823 1.64493406684823 0.657973626739291 0.65797362673929


smallest |zeta(2+it)| found: (mpf('0.66917745288271202'), 15387.44)


**Problem 2.17 (almost periods).** The $\tau\le10^5$ with $|\zeta(2+i\tau)-\zeta(2)|<0.2$ (computed with $n<200$; the tail changes the difference by at most $0.01$): they come in short clusters, with
bounded gaps between the clusters.

In [16]:
n = np.arange(1, 200.); ln = np.log(n); w_ = n ** -2.0; z2 = w_.sum()
res = []
for start in np.arange(1, 1e5, 1e4):
    t = np.arange(start, start + 1e4, 0.01); v = np.zeros(len(t), complex)
    for k in range(len(n)):
        v += w_[k] * np.exp(-1j * t * ln[k])
    res += list(t[abs(v - z2) < 0.2])
res = np.array(res); clusters = res[np.r_[True, np.diff(res) > 0.5]]
print(len(clusters), "clusters; the first ones near", np.round(clusters[:6], 1), " largest gap:", round(np.diff(clusters).max(), 1))

3544 clusters; the first ones near [ 45.4  63.   90.7 108.6 136.3 154.2]  largest gap: 63.3


**Problem 2.18 (van de Lune).** $\sigma_0$ from the prime zeta function, the size of the tail $\int_x^\infty\frac{t^{-\sigma_0}}{\log t}\dd t=E_1((\sigma_0-1)\log x)$ for $x=10^{12}$, $10^{22}$, and the values
of $A$ truncated to $p<10^4$ (used in Figure 2.3).

In [17]:
mp.mp.dps = 30
A = lambda s: mp.nsum(lambda k: mp.binomial(2 * k, k) / (4 ** k * (2 * k + 1)) * mp.primezeta((2 * k + 1) * s), [0, mp.inf])
s0 = mp.findroot(lambda s: A(s) - mp.pi / 2, 1.19)
print("sigma_0 =", s0)
for x in (1e12, 1e22):
    print(x, mp.e1((s0 - 1) * mp.log(x)))
P = list(primerange(2, 10 ** 4))
print({s: sum(math.asin(p ** -s) for p in P) for s in (1.1, 1.19234, 1.5)}, " pi/2 =", pi / 2)
mp.mp.dps = 20

sigma_0 = 1.19234733718619320289750442743
1000000000000.0 0.000794834602769314700495907661011
1e+22 0.000005501234192329121455957005092
{1.1: 1.8830899718691074, 1.19234: 1.503607681010003, 1.5: 0.8569146193433814}  pi/2 = 1.5707963267948966


**Problem 2.19 (Khinchin).** For the rotation by $\sqrt2$ and $A=[0,0.3[$, the average of $\mu(A\cap T^{-n}A)=\max(0,a-\|n\alpha\|)$ tends to $a^2=0.09$.

In [18]:
a, al = 0.3, sqrt(2)
for N in (10 ** 3, 10 ** 5):
    n = np.arange(N); d = np.abs((n * al + 0.5) % 1 - 0.5)
    print(N, np.mean(np.maximum(0, a - d)))

1000 0.09012019998464288
100000 0.09000178609138154


**Problem 2.20 (no rate).** With $a_N=\frac1{\log(N+2)}$, the function $g$ of the solution satisfies $\|A_Ng\|\ge a_N$. We compute $\|A_Ng\|^2=\int|D_N|^2|g|^2$ with the mass function $F$.

In [19]:
aN = lambda N: 1 / log(N + 2)
c = pi ** 2 / 4
def F(r):                                   # piecewise linear through (1/(2N), c a_N^2)
    if r >= 0.5: return c * aN(1) ** 2
    N = math.floor(1 / (2 * r)); r1, r0 = 1 / (2 * N), 1 / (2 * (N + 1))
    return c * aN(N + 1) ** 2 + (c * aN(N) ** 2 - c * aN(N + 1) ** 2) * (r - r0) / (r1 - r0)
def normAN(N, K=200000):
    r = np.linspace(1e-7, 0.5, K); Fr = np.array([F(x) for x in r]); dF = np.diff(Fr)
    mid = (r[1:] + r[:-1]) / 2
    D2 = (np.sin(pi * N * mid) / (N * np.sin(pi * mid))) ** 2
    return sqrt(np.sum(D2 * dF) + F(1e-7))
for N in (1, 5, 50, 500):
    print(N, normAN(N), aN(N))

1 1.4298004336900636 0.9102392266268373


5 0.7923730832317218 0.5138983423697507


50 0.39068256820222846 0.2530848692773169


500 0.24780439911720692 0.16080789578886323


**Problem 2.21 (Wiener).** The Cantor measure: $|\hat\mu(1)|=\prod_{k\ge1}|\cos\frac{2\pi}{3^k}|=0.37143\ldots$, and the averages $\frac1{2N+1}\sum_{|n|\le N}|\hat\mu(n)|^2$ decrease to $0$, slowly.

In [20]:
def muhat(t, K=40):
    return np.prod([np.cos(2 * pi * t / 3 ** k) for k in range(1, K)], axis=0)
print(abs(muhat(1.0)), abs(muhat(27.0)))
for N in (10, 100, 1000, 10000):
    n = np.arange(-N, N + 1, dtype=float)
    print(N, np.mean(muhat(n) ** 2))

0.3714373567087654 0.3714373567087654
10 0.1014790761773114
100 0.02215838720911041
1000 0.005125642909540918
10000 0.001314545615383072


**Problem 2.22 (Gebelein).** Gauss--Hermite quadrature: $\mathbb E[\mathrm{He}_m\mathrm{He}_n]=n!\,\delta_{mn}$; Monte Carlo: $\operatorname{corr}(X^2,Y^2)=\rho^2$ and $\operatorname{corr}(X^3,Y^3)$, which is not $\rho^3$ since $X^3$ has a
component on $\mathrm{He}_1$.

In [21]:
from numpy.polynomial import hermite_e as He
x, w = He.hermegauss(40); w = w / w.sum()
Hm = lambda m: He.hermeval(x, [0] * m + [1])
print(np.round([[np.sum(w * Hm(m) * Hm(n)) for n in range(5)] for m in range(5)], 8))
rho = 0.6; X = rng.normal(size=10 ** 6); Y = rho * X + sqrt(1 - rho ** 2) * rng.normal(size=10 ** 6)
print("corr(X^2,Y^2) =", np.corrcoef(X ** 2, Y ** 2)[0, 1], " rho^2 =", rho ** 2)
# X^3 = He_3 + 3 He_1, so corr = (6 rho^3 + 9 rho) / 15; exact quadrature with Y = rho X + sqrt(1-rho^2) Z
EXY = sum(wi * wj * xi ** 3 * (rho * xi + sqrt(1 - rho ** 2) * xj) ** 3 for xi, wi in zip(x, w) for xj, wj in zip(x, w))
print("corr(X^3,Y^3): quadrature", EXY / 15, " Monte Carlo", np.corrcoef(X ** 3, Y ** 3)[0, 1], " predicted", (6 * rho ** 3 + 9 * rho) / 15)

[[ 1.  0.  0.  0. -0.]
 [ 0.  1.  0. -0.  0.]
 [ 0. -0.  2.  0. -0.]
 [ 0. -0.  0.  6.  0.]
 [-0.  0. -0.  0. 24.]]
corr(X^2,Y^2) = 0.36268645674092675  rho^2 = 0.36


corr(X^3,Y^3): quadrature 0.4464000000000002  Monte Carlo 0.44954794267353515  predicted 0.44639999999999996


## Part B: laboratory

**Experiment 1 (the Nyman--Beurling distance).** By Problem 2.15, in the variable $u=\frac1x$ everything is a sum over $n$: with $f_k(n)=\frac{n\bmod k}k$,
$\langle\rho_{1/j},\rho_{1/k}\rangle=\sum_n\frac{f_j(n)f_k(n)}{n(n+1)}$ and $\langle\mathbf 1,\rho_{1/k}\rangle=\sum_n\frac{f_k(n)}{n(n+1)}$. So the distance $d_N$ from $\mathbf 1$ to the span of $\rho_{1/k}$, $2\le k\le N$
(note $\rho_1=0$), is $d_N^2=1-b^{\mathsf T}G^{-1}b$.

*Exercise 1.* Compute $d_N$ for $N\le120$ and compare $d_N^2\log N$ with $2+\gamma-\log4\pi=0.0461\ldots$, the constant conjectured to be its limit (Báez-Duarte, Balazard, Landreau and Saias, 2000).

**Experiment 2 (the constant in the mean value theorem).** Problem 2.6 gives $T+O(\sqrt{TN})$; Montgomery and Vaughan proved $T+O(N)$ for the largest eigenvalue $\lambda_N(T)$ of the matrix
$\bigl(\int_0^T(m/n)^{it}\dd t\bigr)_{m,n\le N}$.

*Exercise 2.* For $N=30$, plot $\lambda_N(T)-T$ for $T$ from $N$ to $40N$. Does it look bounded? Compare with $2\pi N$.

**Experiment 3 (Hermite expansions).** By Problem 2.22, $\mathbb E[f(X)g(Y)]=\sum_n\rho^nf_ng_n$.

*Exercise 3.* For $f=g=\mathbf 1_{x>0}-\frac12$, compute the Hermite coefficients $f_n$ (use $(\mathrm{He}_{n-1}\varphi)'=-\mathrm{He}_n\varphi$, where $\varphi$ is the Gaussian density), sum the series, and compare with the exact value
$P(X>0,Y>0)-\frac14=\frac{\arcsin\rho}{2\pi}$ (Sheppard's formula).

### Solutions

In [22]:
# Exercise 1
M = 100000
nn = np.arange(1, M + 1); wgt = 1.0 / (nn * (nn + 1.0))
Nmax = 120
Fk = np.array([(nn % k) / k for k in range(2, Nmax + 1)])
# the tail n > M is approximated by the mean of f_j f_k over n <= M, times sum_{n>M} 1/(n(n+1)) = 1/(M+1)
Gfull = (Fk * wgt) @ Fk.T + (Fk @ Fk.T / M) / (M + 1)
bfull = Fk @ wgt + Fk.mean(1) / (M + 1)
for N in (10, 30, 60, 120):
    G_ = Gfull[:N - 1, :N - 1]; b_ = bfull[:N - 1]
    d2 = 1 - b_ @ np.linalg.solve(G_, b_)
    print(N, "d_N =", sqrt(d2), " d_N^2 log N =", d2 * log(N))
print("2+gamma-log(4pi) =", 2 + float(mp.euler) - log(4 * pi))

10 d_N = 0.1544427182734326  d_N^2 log N = 0.0549225334919188
30 d_N = 0.12065128932603451  d_N^2 log N = 0.04951032426040999
60 d_N = 0.10717945910636065  d_N^2 log N = 0.04703352298066629
120 d_N = 0.09868058103680481  d_N^2 log N = 0.04661991033302445
2+gamma-log(4pi) = 0.04619141793224202


The products $d_N^2\log N$ drift slowly towards the conjectured constant: the approximation of $\mathbf 1$ improves only like $1/\sqrt{\log N}$, as Burnol's lower bound says it must.

In [23]:
# Exercise 2
N = 30
L = np.log(np.arange(1, N + 1)); dl = L[:, None] - L[None, :]
for T in (30, 120, 300, 600, 1200):
    with np.errstate(invalid="ignore", divide="ignore"):
        M = np.where(dl == 0, T, (np.exp(1j * dl * T) - 1) / (1j * dl))
    lm = np.linalg.eigvalsh((M + M.conj().T) / 2).max()
    print(T, round(lm - T, 2), " 2 pi N =", round(2 * pi * N, 1))

30 127.25  2 pi N = 188.5
120 68.32  2 pi N = 188.5
300 86.42  2 pi N = 188.5
600 99.8  2 pi N = 188.5
1200 95.56  2 pi N = 188.5


The excess $\lambda_N(T)-T$ stays bounded as $T$ grows, of the order of $N$ and below $2\pi N$: the factor $T+O(N)$ of Montgomery and Vaughan, not $T+O(\sqrt{TN})$, is the truth for large $T$.

In [24]:
# Exercise 3: E[f He_n] = int_0^oo He_n phi = phi(0) He_{n-1}(0) for n >= 1, and He_{2k}(0) = (-1)^k (2k-1)!!
mp.mp.dps = 30
phi0 = 1 / mp.sqrt(2 * mp.pi)
for rho in (0.3, 0.6, 0.9, 0.99):
    # only n = 2k+1 contributes: rho^n f_n^2 = rho^(2k+1) phi(0)^2 ((2k-1)!!)^2 / (2k+1)!, summed by a recurrence
    t = mp.mpf(rho) * phi0 ** 2; series = 0
    for k in range(20000):
        series += t; t *= mp.mpf(rho) ** 2 * (2 * k + 1) ** 2 / ((2 * k + 2) * (2 * k + 3))
    print(rho, series, mp.asin(rho) / (2 * mp.pi))
mp.mp.dps = 20

0.3 0.0484933420103391433983905466773 0.0484933420103391433983905466773


0.6 0.102416382349566721407161866187 0.102416382349566721407161866187


0.9 0.178216853435646872731965572761 0.178216853435646872731965572761


0.99 0.227473293177793938046564223354 0.227473293177793938046564223354


The series reproduces Sheppard's formula. Only odd $n$ contribute, and $f_n^2=\frac{(n-2)!!^2}{2\pi\,n!}$ decays only like $n^{-3/2}$, because $f$ jumps at $0$: the convergence is slow for $\rho$ close to $1$.
In fact $\sum_{k\ge0}\frac{(2k-1)!!^2}{(2k+1)!}\rho^{2k+1}=\arcsin\rho$ is the Taylor series of $\arcsin$, so the exercise is a proof of Sheppard's formula.